In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -q datasets sentence-transformers accelerate

In [3]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

In [4]:
raw_train = load_dataset("csv", data_files=TRAIN_PATH)["train"]

def make_combined(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

raw_train = raw_train.map(make_combined)

q1_len = len(raw_train[51]["combined_text"])
print("Q1 - combined_text length at index 51:", q1_len)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Q1 - combined_text length at index 51: 614


In [5]:
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
q2_vocab = bert_tok.vocab_size
print("Q2 - bert-base-uncased vocab size:", q2_vocab)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q2 - bert-base-uncased vocab size: 30522


In [6]:
q3_sep_id = bert_tok.sep_token_id
print("Q3 - [SEP] token id:", q3_sep_id)

Q3 - [SEP] token id: 102


In [7]:
all_prompts = [str(p) for p in raw_train["prompt"]]  # cast to plain list[str] to avoid tokenizer input-type errors
encoded_all = bert_tok(
    all_prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)
q4_shape = tuple(encoded_all["input_ids"].shape)
print("Q4 - input_ids shape:", q4_shape)

Q4 - input_ids shape: (2000, 128)


In [8]:
hidden_size = 768
n_heads = 12
q5_head_dim = hidden_size // n_heads
print("Q5 - per-head dimension:", q5_head_dim)

Q5 - per-head dimension: 64


In [9]:
bert_model = AutoModel.from_pretrained("bert-base-uncased")
bert_model.eval()

row0_prompt = raw_train[0]["prompt"]
inputs_row0 = bert_tok(row0_prompt, return_tensors="pt")

with torch.no_grad():
    out_row0 = bert_model(**inputs_row0)

q6_shape = tuple(out_row0.last_hidden_state.shape)
print("Q6 - last_hidden_state shape (row 0):", q6_shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q6 - last_hidden_state shape (row 0): (1, 31, 768)


In [10]:
cls_vec = out_row0.last_hidden_state[0, 0, :]
q7_sum = round(cls_vec[:5].sum().item(), 4)
print("Q7 - sum of first 5 CLS values:", q7_sum)

Q7 - sum of first 5 CLS values: -1.2001


In [11]:
attn_model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
attn_model.eval()

sentence = "Light-ion fusion is a technique."
attn_inputs = bert_tok(sentence, return_tensors="pt")

with torch.no_grad():
    attn_out = attn_model(**attn_inputs)

tokens = bert_tok.convert_ids_to_tokens(attn_inputs["input_ids"][0])
print("tokens:", tokens)
fusion_idx = tokens.index("fusion")

last_layer_attn = attn_out.attentions[-1]
q8_weight = round(last_layer_attn[0, 0, 0, fusion_idx].item(), 4)
print("Q8 - attention weight CLS -> fusion:", q8_weight)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Q8 - attention weight CLS -> fusion: 0.1025


In [12]:
minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
row0 = raw_train[0]
emb_prompt = minilm.encode(row0["prompt"], convert_to_tensor=True)
emb_optB = minilm.encode(row0["B"], convert_to_tensor=True)
q9_sim = round(util.cos_sim(emb_prompt, emb_optB).item(), 4)
print("Q9 - cosine sim (prompt vs option B, row 0):", q9_sim)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Q9 - cosine sim (prompt vs option B, row 0): 0.7658


In [13]:
options_cols = ["A", "B", "C", "D", "E"]

def map_at_3(top3_labels, true_answer):
    if true_answer not in top3_labels:
        return 0.0
    rank = top3_labels.index(true_answer) + 1
    return 1.0 / rank

prompts = raw_train["prompt"]
answers = raw_train["answer"]
option_texts = {c: raw_train[c] for c in options_cols}

tfidf_scores = []
tfidf_top3_all = []

vectorizer = TfidfVectorizer(stop_words="english")
corpus = list(prompts)
for c in options_cols:
    corpus += list(option_texts[c])
vectorizer.fit(corpus)

n = len(raw_train)
prompt_vecs = vectorizer.transform(prompts)
option_vecs = {c: vectorizer.transform(option_texts[c]) for c in options_cols}

for i in range(n):
    sims = {}
    for c in options_cols:
        sims[c] = cosine_similarity(prompt_vecs[i], option_vecs[c][i])[0][0]
    ranked = sorted(sims, key=sims.get, reverse=True)[:3]
    tfidf_top3_all.append(ranked)
    tfidf_scores.append(map_at_3(ranked, answers[i]))

tfidf_map3 = np.mean(tfidf_scores)
print("TF-IDF MAP@3:", round(tfidf_map3, 4))

minilm_scores = []
minilm_top3_all = []

prompt_embs = minilm.encode(list(prompts), convert_to_tensor=True, show_progress_bar=True)
option_embs = {c: minilm.encode(list(option_texts[c]), convert_to_tensor=True, show_progress_bar=True) for c in options_cols}

for i in range(n):
    sims = {}
    for c in options_cols:
        sims[c] = util.cos_sim(prompt_embs[i], option_embs[c][i]).item()
    ranked = sorted(sims, key=sims.get, reverse=True)[:3]
    minilm_top3_all.append(ranked)
    minilm_scores.append(map_at_3(ranked, answers[i]))

minilm_map3 = np.mean(minilm_scores)
print("Q10a - MiniLM MAP@3:", round(minilm_map3, 4))

recovered = 0
for i in range(n):
    tfidf_hit = answers[i] in tfidf_top3_all[i]
    minilm_hit = answers[i] in minilm_top3_all[i]
    if (not tfidf_hit) and minilm_hit:
        recovered += 1

print("Q10b - # questions TF-IDF missed but MiniLM caught:", recovered)

TF-IDF MAP@3: 0.3119


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Q10a - MiniLM MAP@3: 0.4231
Q10b - # questions TF-IDF missed but MiniLM caught: 462


In [14]:
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row1 = raw_train[1]
candidates = [row1["A"], row1["B"], row1["C"]]

zsc_result = zsc(row1["prompt"], candidates)
q11_top_score = round(zsc_result["scores"][0], 4)
print("Q11 - top zero-shot score (softmax):", q11_top_score)
print(zsc_result)

zsc_result_multi = zsc(row1["prompt"], candidates, multi_label=True)
q12_diff = round(abs(sum(zsc_result["scores"]) - sum(zsc_result_multi["scores"])), 4)
print("Q12 - |sum(softmax) - sum(sigmoid)|:", q12_diff)
print(zsc_result_multi)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q11 - top zero-shot score (softmax): 0.4575
{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kin

In [15]:
row0 = raw_train[0]
prompt_str = (
    f"Question: {row0['prompt']}. Is the correct answer "
    f"A: {row0['A']} or B: {row0['B']}? Answer with just the letter A or B."
)
try:
    gen = pipeline("text2text-generation", model="google/flan-t5-small")
    gen_out = gen(prompt_str, max_new_tokens=5)
    q13_output = gen_out[0]["generated_text"]
except KeyError:
    from transformers import AutoModelForSeq2SeqLM
    flan_tok = AutoTokenizer.from_pretrained("google/flan-t5-small")
    flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
    flan_model.eval()
    flan_inputs = flan_tok(prompt_str, return_tensors="pt")
    with torch.no_grad():
        flan_out_ids = flan_model.generate(**flan_inputs, max_new_tokens=5)
    q13_output = flan_tok.decode(flan_out_ids[0], skip_special_tokens=True)

print("Q13 - flan-t5-small output:", repr(q13_output))

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Q13 - flan-t5-small output: 'B'
